In [2]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

In [3]:
import sys
sys.path.append("..") 

In [4]:
from processing import load_UCI_dataset,extract_all_features,extract_features_from_dataset, select_f_test, select_mrmr, select_reliefF, save_feature_set

In [5]:
X_train, y_train, X_val, y_val, X_test, y_test = load_UCI_dataset(WINDOW_SIZE= 625, STEP_SIZE= 312)

Total recordings: 12000
Train recordings: 9600
Validation recordings: 1200
Test recordings: 1200


100%|██████████| 9600/9600 [02:33<00:00, 62.70it/s]


Skipped recordings: 81


100%|██████████| 1200/1200 [00:20<00:00, 59.31it/s]


Skipped recordings: 7


100%|██████████| 1200/1200 [00:19<00:00, 60.61it/s]


Skipped recordings: 9


In [6]:
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print("X_train min:", X_train.min())
print("X_train max:", X_train.max())

X_train shape: (829483, 1, 625), y_train shape: (829483, 2)
X_train min: 0.0
X_train max: 1.0


## Feature Exraction - Non-fiducial features ##

We'll extract 19 features

## Feature Exraction - Non-fiducial features ##

We'll extract 19 features

In [7]:
test_window = X_train[0, 0, :]

features = extract_all_features(
    test_window,
    fs=125
)

print("Number of features:", len(features))

print(features)

Number of features: 57
{'ppg_mean': np.float64(0.5528637123954845), 'ppg_median': np.float64(0.5925931433156838), 'ppg_std': np.float64(0.2905845371580045), 'ppg_variance': np.float64(0.08443937323533167), 'ppg_iqr': np.float64(0.5234703093409055), 'ppg_skewness': np.float64(-0.22736716400990045), 'ppg_kurtosis': np.float64(-1.2339547439481824), 'ppg_zero_crossing_rate': np.float64(0.0), 'ppg_shannon_entropy': np.float64(5.574628051267654), 'ppg_energy_mean': np.float64(0.3900976577190485), 'ppg_energy_variance': np.float64(0.09636289363250157), 'ppg_energy_skewness': np.float64(0.3190031184262097), 'ppg_energy_kurtosis': np.float64(-1.2737042691425176), 'ppg_energy_iqr': np.float64(0.5817711450048391), 'ppg_kte_mean': np.float64(0.0015399260317650507), 'ppg_kte_variance': np.float64(4.096121882933001e-06), 'ppg_kte_skewness': np.float64(1.1982202259142651), 'ppg_kte_kurtosis': np.float64(1.2361426030677016), 'ppg_kte_iqr': np.float64(0.002613196788427148), 'vpg_mean': np.float64(-0.00

In [8]:
df_train = extract_features_from_dataset(
    X_train,
    y_train
)

df_val = extract_features_from_dataset(
    X_val,
    y_val
)

df_test = extract_features_from_dataset(
    X_test,
    y_test
)

In [9]:
print("Train:", df_train.shape)
print("Validation:", df_val.shape)
print("Test:", df_test.shape)

Train: (829483, 59)
Validation: (104736, 59)
Test: (105176, 59)


## Feature Selection ##

We'll reduce the 57 features to 15 using 3 different methods

In [10]:
feature_columns = [
    col for col in df_train.columns
    if col not in ["SBP", "DBP"]
]

X_train = df_train[feature_columns]
X_val = df_val[feature_columns]
X_test = df_test[feature_columns]

y_train_sbp = df_train["SBP"]
y_train_dbp = df_train["DBP"]

y_val_sbp = df_val["SBP"]
y_val_dbp = df_val["DBP"]

y_test_sbp = df_test["SBP"]
y_test_dbp = df_test["DBP"]

## F-Test ##

In [11]:
# SBP
f_sbp_features, f_sbp_results = select_f_test(
    X_train,
    y_train_sbp,
    k=15
)
print("F-test selected features for SBP:")

for i, feature in enumerate(f_sbp_features, 1):
    print(i, feature)

print(f_sbp_results)

F-test selected features for SBP:
1 vpg_iqr
2 vpg_skewness
3 vpg_energy_iqr
4 apg_zero_crossing_rate
5 apg_energy_iqr
6 ppg_iqr
7 vpg_shannon_entropy
8 apg_skewness
9 apg_shannon_entropy
10 vpg_kte_iqr
11 apg_iqr
12 vpg_energy_skewness
13 ppg_kte_iqr
14 vpg_kurtosis
15 ppg_energy_iqr
                   feature       F_score        p_value
0                  vpg_iqr  38942.523886   0.000000e+00
1             vpg_skewness  32611.968460   0.000000e+00
2           vpg_energy_iqr  29749.677227   0.000000e+00
3   apg_zero_crossing_rate  29436.898731   0.000000e+00
4           apg_energy_iqr  20116.611075   0.000000e+00
5                  ppg_iqr  18009.693634   0.000000e+00
6      vpg_shannon_entropy  17461.529221   0.000000e+00
7             apg_skewness  16953.696172   0.000000e+00
8      apg_shannon_entropy  16482.853549   0.000000e+00
9              vpg_kte_iqr  15954.574964   0.000000e+00
10                 apg_iqr  14605.661375   0.000000e+00
11     vpg_energy_skewness  14004.177510   

In [12]:
# DBP
f_dbp_features, f_dbp_results = select_f_test(
    X_train,
    y_train_dbp,
    k=15
)

print("\nF-test selected features for DBP:")

for i, feature in enumerate(f_dbp_features, 1):
    print(i, feature)

print(f_dbp_results)


F-test selected features for DBP:
1 vpg_median
2 apg_iqr
3 apg_kte_iqr
4 ppg_mean
5 apg_kte_mean
6 apg_std
7 ppg_median
8 apg_median
9 vpg_kte_mean
10 apg_variance
11 apg_energy_mean
12 ppg_skewness
13 ppg_shannon_entropy
14 ppg_energy_skewness
15 ppg_kte_iqr
                   feature       F_score        p_value
0               vpg_median  19574.555545   0.000000e+00
1                  apg_iqr  19231.519586   0.000000e+00
2              apg_kte_iqr  18398.025323   0.000000e+00
3                 ppg_mean  17929.096899   0.000000e+00
4             apg_kte_mean  15378.280929   0.000000e+00
5                  apg_std  15086.272859   0.000000e+00
6               ppg_median  14979.571962   0.000000e+00
7               apg_median  14656.588855   0.000000e+00
8             vpg_kte_mean  13893.913572   0.000000e+00
9             apg_variance  12614.432097   0.000000e+00
10         apg_energy_mean  12598.309909   0.000000e+00
11            ppg_skewness  11398.696637   0.000000e+00
12     ppg_

## mRMR ##

In [13]:
mrmr_sbp_features, mrmr_sbp_results = select_mrmr(
    X_train,
    y_train_sbp,
    k=15
)

print("mRMR selected features for SBP:")

for i, feature in enumerate(
    mrmr_sbp_features,
    1
):
    print(i, feature)

print(mrmr_sbp_results)

mRMR: using 50000 samples for feature selection
mRMR selected features for SBP:
1 vpg_iqr
2 ppg_zero_crossing_rate
3 apg_energy_variance
4 ppg_energy_kurtosis
5 vpg_mean
6 ppg_std
7 vpg_median
8 apg_energy_kurtosis
9 apg_median
10 apg_mean
11 apg_zero_crossing_rate
12 ppg_median
13 ppg_kte_iqr
14 vpg_skewness
15 apg_skewness
                   feature  relevance_MI  avg_redundancy  mRMR_score  \
0                  vpg_iqr      0.052428        0.000000    0.052428   
1   ppg_zero_crossing_rate      0.000000        0.000000    0.000000   
2      apg_energy_variance      0.037574        0.020724    0.016850   
3      ppg_energy_kurtosis      0.043918        0.034434    0.009484   
4                 vpg_mean      0.003366        0.006892   -0.003526   
5                  ppg_std      0.013085        0.033228   -0.020143   
6               vpg_median      0.043720        0.060308   -0.016588   
7      apg_energy_kurtosis      0.044859        0.066399   -0.021541   
8               apg_media

In [14]:
mrmr_dbp_features, mrmr_dbp_results = select_mrmr(
    X_train,
    y_train_dbp,
    k=15
)

print("\nmRMR selected features for DBP:")

for i, feature in enumerate(
    mrmr_dbp_features,
    1
):
    print(i, feature)


print(mrmr_dbp_results)

mRMR: using 50000 samples for feature selection

mRMR selected features for DBP:
1 apg_energy_variance
2 vpg_median
3 ppg_zero_crossing_rate
4 ppg_std
5 vpg_mean
6 apg_median
7 ppg_mean
8 apg_mean
9 ppg_kte_kurtosis
10 apg_zero_crossing_rate
11 apg_kte_iqr
12 vpg_skewness
13 ppg_energy_iqr
14 apg_skewness
15 apg_iqr
                   feature  relevance_MI  avg_redundancy  mRMR_score  \
0      apg_energy_variance      0.044653        0.000000    0.044653   
1               vpg_median      0.033682        0.024705    0.008978   
2   ppg_zero_crossing_rate      0.000000        0.000000    0.000000   
3                  ppg_std      0.009697        0.007514    0.002182   
4                 vpg_mean      0.002574        0.006472   -0.003898   
5               apg_median      0.021719        0.032688   -0.010969   
6                 ppg_mean      0.042324        0.072157   -0.029833   
7                 apg_mean      0.001219        0.029825   -0.028606   
8         ppg_kte_kurtosis      0.

## ReliefF ##

In [15]:
relief_sbp_features, relief_sbp_results = select_reliefF(
    X_train,
    y_train_sbp,
    k=15
)

print("ReliefF selected features for SBP:")

for i, feature in enumerate(
    relief_sbp_features,
    1
):
    print(i, feature)


print(relief_sbp_results)

RReliefF: using 50000 samples for feature selection
ReliefF selected features for SBP:
1 vpg_mean
2 apg_mean
3 apg_zero_crossing_rate
4 apg_median
5 vpg_median
6 apg_skewness
7 ppg_energy_variance
8 vpg_skewness
9 vpg_shannon_entropy
10 vpg_zero_crossing_rate
11 ppg_variance
12 ppg_std
13 ppg_shannon_entropy
14 ppg_kte_iqr
15 vpg_iqr
                   feature  RReliefF_score
0                 vpg_mean        0.104692
1                 apg_mean        0.098696
2   apg_zero_crossing_rate        0.090594
3               apg_median        0.077161
4               vpg_median        0.072071
5             apg_skewness        0.069405
6      ppg_energy_variance        0.069029
7             vpg_skewness        0.067353
8      vpg_shannon_entropy        0.065285
9   vpg_zero_crossing_rate        0.065190
10            ppg_variance        0.062887
11                 ppg_std        0.062308
12     ppg_shannon_entropy        0.061835
13             ppg_kte_iqr        0.061771
14                 

In [16]:
relief_dbp_features, relief_dbp_results = select_reliefF(
    X_train,
    y_train_dbp,
    k=15
)

print("\nReliefF selected features for DBP:")

for i, feature in enumerate(
    relief_dbp_features,
    1
):
    print(i, feature)
    
print(relief_dbp_results)

RReliefF: using 50000 samples for feature selection

ReliefF selected features for DBP:
1 vpg_mean
2 apg_mean
3 apg_zero_crossing_rate
4 apg_median
5 vpg_median
6 ppg_energy_variance
7 apg_skewness
8 vpg_skewness
9 vpg_shannon_entropy
10 vpg_zero_crossing_rate
11 ppg_variance
12 ppg_std
13 ppg_kte_iqr
14 vpg_iqr
15 ppg_shannon_entropy
                   feature  RReliefF_score
0                 vpg_mean        0.063275
1                 apg_mean        0.060428
2   apg_zero_crossing_rate        0.053940
3               apg_median        0.047499
4               vpg_median        0.043723
5      ppg_energy_variance        0.041962
6             apg_skewness        0.041421
7             vpg_skewness        0.040429
8      vpg_shannon_entropy        0.038844
9   vpg_zero_crossing_rate        0.038660
10            ppg_variance        0.038099
11                 ppg_std        0.037816
12             ppg_kte_iqr        0.037469
13                 vpg_iqr        0.037387
14     ppg_shannon

In [17]:
selection_summary = {

    "SBP_F_test": f_sbp_features,

    "SBP_mRMR": mrmr_sbp_features,

    "SBP_ReliefF": relief_sbp_features,

    "DBP_F_test": f_dbp_features,

    "DBP_mRMR": mrmr_dbp_features,

    "DBP_ReliefF": relief_dbp_features
}

In [18]:
for name, features in selection_summary.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    for i, feature in enumerate(features, 1):
        print(f"{i:2d}. {feature}")


SBP_F_test
 1. vpg_iqr
 2. vpg_skewness
 3. vpg_energy_iqr
 4. apg_zero_crossing_rate
 5. apg_energy_iqr
 6. ppg_iqr
 7. vpg_shannon_entropy
 8. apg_skewness
 9. apg_shannon_entropy
10. vpg_kte_iqr
11. apg_iqr
12. vpg_energy_skewness
13. ppg_kte_iqr
14. vpg_kurtosis
15. ppg_energy_iqr

SBP_mRMR
 1. vpg_iqr
 2. ppg_zero_crossing_rate
 3. apg_energy_variance
 4. ppg_energy_kurtosis
 5. vpg_mean
 6. ppg_std
 7. vpg_median
 8. apg_energy_kurtosis
 9. apg_median
10. apg_mean
11. apg_zero_crossing_rate
12. ppg_median
13. ppg_kte_iqr
14. vpg_skewness
15. apg_skewness

SBP_ReliefF
 1. vpg_mean
 2. apg_mean
 3. apg_zero_crossing_rate
 4. apg_median
 5. vpg_median
 6. apg_skewness
 7. ppg_energy_variance
 8. vpg_skewness
 9. vpg_shannon_entropy
10. vpg_zero_crossing_rate
11. ppg_variance
12. ppg_std
13. ppg_shannon_entropy
14. ppg_kte_iqr
15. vpg_iqr

DBP_F_test
 1. vpg_median
 2. apg_iqr
 3. apg_kte_iqr
 4. ppg_mean
 5. apg_kte_mean
 6. apg_std
 7. ppg_median
 8. apg_median
 9. vpg_kte_mean
10

## Saving ##

In [19]:
save_feature_set(
    name="SBP_FTEST",
    selected_features=f_sbp_features,
    target="SBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="SBP_MRMR",
    selected_features=mrmr_sbp_features,
    target="SBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="SBP_RELIEFF",
    selected_features=relief_sbp_features,
    target="SBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

Processing: SBP_FTEST
Number of selected features: 15
Train shape: (829483, 16)
Val shape:   (104736, 16)
Test shape:  (105176, 16)

Saved:
data/SBP_FTEST_train.csv
data/SBP_FTEST_val.csv
data/SBP_FTEST_test.csv
data/SBP_FTEST_preprocessing.pkl

Processing: SBP_MRMR
Number of selected features: 15
Train shape: (829483, 16)
Val shape:   (104736, 16)
Test shape:  (105176, 16)

Saved:
data/SBP_MRMR_train.csv
data/SBP_MRMR_val.csv
data/SBP_MRMR_test.csv
data/SBP_MRMR_preprocessing.pkl

Processing: SBP_RELIEFF
Number of selected features: 15
Train shape: (829483, 16)
Val shape:   (104736, 16)
Test shape:  (105176, 16)

Saved:
data/SBP_RELIEFF_train.csv
data/SBP_RELIEFF_val.csv
data/SBP_RELIEFF_test.csv
data/SBP_RELIEFF_preprocessing.pkl



In [20]:
save_feature_set(
    name="DBP_FTEST",
    selected_features=f_dbp_features,
    target="DBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="DBP_MRMR",
    selected_features=mrmr_dbp_features,
    target="DBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

save_feature_set(
    name="DBP_RELIEFF",
    selected_features=relief_dbp_features,
    target="DBP",
    df_train=df_train,
    df_val=df_val,
    df_test=df_test
)

Processing: DBP_FTEST
Number of selected features: 15
Train shape: (829483, 16)
Val shape:   (104736, 16)
Test shape:  (105176, 16)

Saved:
data/DBP_FTEST_train.csv
data/DBP_FTEST_val.csv
data/DBP_FTEST_test.csv
data/DBP_FTEST_preprocessing.pkl

Processing: DBP_MRMR
Number of selected features: 15
Train shape: (829483, 16)
Val shape:   (104736, 16)
Test shape:  (105176, 16)

Saved:
data/DBP_MRMR_train.csv
data/DBP_MRMR_val.csv
data/DBP_MRMR_test.csv
data/DBP_MRMR_preprocessing.pkl

Processing: DBP_RELIEFF
Number of selected features: 15
Train shape: (829483, 16)
Val shape:   (104736, 16)
Test shape:  (105176, 16)

Saved:
data/DBP_RELIEFF_train.csv
data/DBP_RELIEFF_val.csv
data/DBP_RELIEFF_test.csv
data/DBP_RELIEFF_preprocessing.pkl

